# K-Nearest Neighbors (KNN)

Course 3: Supervised Learning, Klasifikasi

KNN memprediksi kelas sebuah data dengan melihat `k` data terdekatnya di data latih, lalu mengambil kelas yang paling banyak muncul di antara tetangga itu. Modelnya sederhana dan mudah dijelaskan. Di notebook ini aku mencoba beberapa nilai `k`, lalu mencari `k` terbaik.

## Dataset

Datanya `tumor.csv`, hasil uji laboratorium dari sampel tumor (misalnya ukuran dan bentuk sel). Targetnya kolom `Class`: `0` untuk tumor jinak (benign) dan `1` untuk ganas (kanker). Jadi ini klasifikasi biner. Dibaca langsung dari link:

https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML241EN-SkillsNetwork/labs/datasets/tumor.csv

## 1. Import dan baca data

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
RS = 123

In [ ]:
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML241EN-SkillsNetwork/labs/datasets/tumor.csv"
tumor = pd.read_csv(URL)
print(tumor.shape)
tumor.head()

In [ ]:
X = tumor.drop(columns="Class")
y = tumor["Class"]
X.describe().loc[["min", "max", "mean"]].round(2)

Semua fitur numerik dengan rentang yang sama (sekitar 1 sampai 10). KNN bergantung pada jarak, jadi biasanya fitur perlu diskalakan. Di sini tidak perlu karena semua fitur sudah berada pada skala yang sama.

In [ ]:
print((y.value_counts(normalize=True) * 100).round(1).rename({0: "jinak (0)", 1: "ganas (1)"}).to_string())
y.value_counts().rename({0: "jinak", 1: "ganas"}).plot.bar(color=["tab:green", "tab:red"], figsize=(5, 3))
plt.ylabel("jumlah")
plt.show()

Kelas jinak sekitar dua kali lipat kelas ganas. Tidak terlalu timpang, jadi masih aman memakai F1 sebagai pembanding.

## 2. Pisahkan data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RS)
print("train:", X_train.shape, "| test:", X_test.shape)

## 3. Fungsi evaluasi

In [ ]:
def evaluasi(y_benar, y_pred):
    return {
        "accuracy": accuracy_score(y_benar, y_pred),
        "precision": precision_score(y_benar, y_pred),
        "recall": recall_score(y_benar, y_pred),
        "f1": f1_score(y_benar, y_pred),
    }

## 4. KNN dengan k = 2

In [ ]:
knn2 = KNeighborsClassifier(n_neighbors=2).fit(X_train, y_train)
pd.Series(evaluasi(y_test, knn2.predict(X_test))).round(4)

Hasilnya sudah sangat baik untuk data ini, artinya sampel tumor yang mirip memang cenderung punya kelas yang sama.

### Latihan: k = 5

In [ ]:
knn5 = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
hasil_k = pd.DataFrame({"k=2": evaluasi(y_test, knn2.predict(X_test)),
                        "k=5": evaluasi(y_test, knn5.predict(X_test))}).T
hasil_k.round(4)

## 5. Mencari nilai k terbaik

Aku coba `k` dari 1 sampai 50. Untuk tiap `k` ada dua ukuran F1:

- **F1 di data test**, untuk melihat performa akhirnya
- **F1 hasil cross validation 5 lipatan di data train**, yang dipakai untuk memilih `k`

`k` sebaiknya dipilih dari cross validation, bukan dari data test, supaya data test tidak ikut menentukan pilihan model.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RS)
baris = []
for k in range(1, 51):
    model = KNeighborsClassifier(n_neighbors=k)
    f1_cv = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1").mean()
    f1_test = f1_score(y_test, model.fit(X_train, y_train).predict(X_test))
    baris.append({"k": k, "F1 cross validation": f1_cv, "F1 test": f1_test})

hasil_k = pd.DataFrame(baris).set_index("k")
hasil_k.round(4).head(10)

In [ ]:
ax = hasil_k.plot(figsize=(10, 4), marker="o", markersize=3)
ax.set_xlabel("jumlah tetangga (k)")
ax.set_ylabel("F1")
ax.set_title("F1 untuk berbagai nilai k")
plt.show()

In [ ]:
k_terbaik = int(hasil_k["F1 cross validation"].idxmax())
print("k terbaik menurut cross validation:", k_terbaik)
print("F1 cross validation:", round(hasil_k.loc[k_terbaik, "F1 cross validation"], 4))
print("F1 test            :", round(hasil_k.loc[k_terbaik, "F1 test"], 4))

Pembacaan grafiknya: `k` yang sangat kecil membuat model terlalu mengikuti data latih (variance tinggi), sedangkan `k` yang terlalu besar membuat batas keputusan terlalu halus sehingga kelas minoritas ikut "tertelan" (bias tinggi). Nilai `k` terbaik biasanya ada di tengah-tengah.

## 6. Model akhir

In [ ]:
knn_akhir = KNeighborsClassifier(n_neighbors=k_terbaik).fit(X_train, y_train)
pd.Series(evaluasi(y_test, knn_akhir.predict(X_test)), name=f"k={k_terbaik}").round(4)

Satu keunggulan KNN: prediksinya bisa dijelaskan dengan menunjukkan tetangga terdekatnya. Berikut contoh untuk satu sampel test:

In [ ]:
sampel = X_test.iloc[[0]]
jarak, idx = knn_akhir.kneighbors(sampel)
tetangga = pd.DataFrame({"index data latih": X_train.index[idx[0]], "jarak": jarak[0].round(2),
                         "kelas": y_train.iloc[idx[0]].values})
print("prediksi:", knn_akhir.predict(sampel)[0], "| kelas sebenarnya:", y_test.iloc[0])
tetangga

## Ringkasan

- KNN mengklasifikasikan lewat voting tetangga terdekat, jadi sensitif terhadap skala fitur dan nilai `k`.
- Nilai `k` dipilih lewat cross validation di data train, bukan lewat data test.
- `k` kecil cenderung overfit dan `k` besar cenderung terlalu sederhana.
- Kelebihan KNN adalah mudah dijelaskan, kekurangannya prediksi jadi lambat untuk data yang sangat besar karena harus menghitung jarak ke semua data latih.